# v15 — 쌍별 분류기 입력 단순화와 학습 전용 증강
raw/gene/type × 증강 없음/평균 희귀 변이 10개의 6개 설정을 비교합니다. C=0.1, empirical LR, k=512, 두 쌍 및 alpha=0.5는 고정합니다. test/sample 접근을 차단하고 홀드아웃은 점수 계산에 사용하지 않습니다. 증강은 라벨 유지가 보장된 생물학적 변환이 아닌 합성 가정입니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
forbidden_attempts15=[]
def forbid_inference15(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:
  forbidden_attempts15.append(str(args[0]));raise RuntimeError('Inference data access is forbidden in v15')
sys.addaudithook(forbid_inference15)
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if (p/'data/train.csv').is_file()),None)
if REPO is None:raise FileNotFoundError('data/train.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v15';OUT.mkdir(parents=True,exist_ok=True)
# Inference files and submission directory are intentionally not used.
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert train.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('TRAIN ONLY',train.shape,'Dev/holdout',len(dev),len(hold),'OUT',OUT,flush=True)

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v15


In [2]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


In [3]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Novel settings:',len(CONFIGS),'Views:',VIEWS)

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Total new configurations after refinement:',len(CONFIGS))


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59


In [4]:
# Freeze the previous version; do not repeat its 59-setting search.
BASE_CONFIGS=copy.deepcopy(CONFIGS)
BASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]

def fit_base(idx):
 cache={};ms=[];ws=[]
 for name,w in BASE_PLAN:
  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)
  else:ms.append(train_new(name,idx,cache));ws.append(w)
 return ms,np.array(ws)


In [5]:
class NewSelector:
 def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}
 def select(self,method,k):
  if (method,k) in self.cache:return self.cache[(method,k)]
  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',RuntimeWarning)
   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)
   if method=='balanced':
    orders=[]
    for c in range(NC):
     yes=self.y==c
     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)
     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()
     score[~high|numeric]=0
     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])
    order=[]
    for rank in range(max(map(len,orders))):
     for o in orders:
      if rank<len(o):order.append(int(o[rank]))
    order+=list(np.argsort(-global_score,kind='stable'))
   else:order=list(np.argsort(-global_score,kind='stable'))
  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()
  for j in order:
   j=int(j)
   if j in seen:continue
   seen.add(j)
   if global_score[j]<=0:continue
   lo,hi=z.indptr[j:j+2]
   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()
   if sig in signatures:continue
   signatures.add(sig);chosen.append(j)
   if len(chosen)>=k:break
  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]

class SpecificityPost:
 def fit(self,a,names,target):
  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
  z=(a[:,self.binary]!=0).astype(np.float32)
  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])
  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)
  posterior=(counts+10*prior[:,None])/(freq+10)
  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)
  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)
  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self
 def transform(self,a):
  from sklearn.preprocessing import normalize
  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')
  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))

NEW={}
for view in ['raw','gated_syn_down_s3']:
 for method in ['global','balanced']:
  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)
for c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)
for c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)

def fit_new6(name,idx,cache):
 cfg=NEW[name];view=cfg['view']
 if view not in cache:
  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))
 enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]
 if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)
 else:post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=estimator(cfg);tx=post.transform(a[:,cols])
 if cfg.get('augment'):
  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]
 else:rw=np.ones(len(idx))
 if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)
 elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 m=Member(name,view,enc,space,cols,post,est)
 # Column order of a specificity block changes, but raw_names always describe the vectorizer input.
 if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])
 return m
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))
print('New configurations',len(NEW))

NEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))

print('Total new configurations including augmentation extension:',len(NEW))


New configurations 8
Total new configurations including augmentation extension: 12


In [6]:
def fit_plan6(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v5_reference':
   old,ow=basefit if basefit is not None else fit_base(idx)
   ms.extend(old);ws.extend(ow*w)
  else:ms.append(fit_new6(name,idx,cache));ws.append(w)
 return ms,np.array(ws)

V6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]
def fit_v6(idx):return fit_plan6(idx,V6_PLAN)


In [7]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_selection import SelectKBest

def expand10(raw):
 t=canon(raw);m=re.fullmatch(r'([0-9]+)_([0-9]+)([A-Z]+)>([A-Z]+)',t)
 if m and int(m[2])-int(m[1])+1==len(m[3])==len(m[4]):
  return tuple(f'{a}{int(m[1])+j}{b}' for j,(a,b) in enumerate(zip(m[3],m[4])))
 return (t,)

def kind10(t):
 if 'fs' in t:return 'frameshift'
 if t.endswith('*'):return 'stop'
 if 'delins' in t:return 'delins'
 if 'del' in t:return 'deletion'
 if 'ins' in t:return 'insertion'
 m=re.fullmatch(r'([A-Z])[0-9]+([A-Z])',t)
 if m:return 'synonymous' if m[1]==m[2] else 'substitution'
 return 'other'

def packet10(frame,ev=None):
 if ev is None:ev=parse_events(frame[genes])
 resolved=[[(g,tuple(sorted({s for t in ts for s in expand10(t)}))) for g,ts in row] for row in ev]
 missing=frame[genes].isna().to_numpy()
 return dict(events=resolved,missing=missing)

def records10(packet,representation,numeric=True):
 rs=[]
 for i,row in enumerate(packet['events']):
  d={};syn=nonsyn=structural=unparsed=0
  for g,ts in row:
   d['gene:'+g]=1.;types={kind10(t) for t in ts}
   sy='synonymous' in types;ns=bool(types.intersection({'substitution','stop','frameshift','deletion','insertion','delins'}));st=bool(types.intersection({'stop','frameshift','deletion','insertion','delins'}));ot='other' in types
   syn+=sy;nonsyn+=ns;structural+=st;unparsed+=ot
   if representation=='type':
    for typ in types:d['type:'+g+':'+typ]=1.
   elif representation=='functional':
    for flag,family in [(sy,'syn_gene'),(ns,'nonsyn_gene'),(st,'structural_gene'),(ot,'unparsed_gene')]:
     if flag:d[family+':'+g]=1.
  if numeric:
   d['numeric:log_observed_mutated_genes']=np.log1p(len(row));d['numeric:log_missing_genes']=np.log1p(int(packet['missing'][i].sum()))
   if representation!='gene':
    for title,value in [('syn',syn),('nonsyn',nonsyn),('structural',structural),('unparsed',unparsed)]:
     d['numeric:log_'+title+'_genes']=np.log1p(value);d['numeric:fraction_'+title+'_genes']=value/max(len(row),1)
  rs.append(d)
 return rs

def mask_packet10(packet,probability,seed):
 rng=np.random.default_rng(seed);out=[];missing=packet['missing'].copy();gpos={g:j for j,g in enumerate(genes)}
 for i,row in enumerate(packet['events']):
  keep=[]
  for g,ts in row:
   if rng.random()<probability:missing[i,gpos[g]]=True
   else:keep.append((g,ts))
  out.append(keep)
 return dict(events=out,missing=missing)

def select_cols10(a,target,k):
 if k is None or a.shape[1]<=k:return np.arange(a.shape[1])
 with warnings.catch_warnings():
  warnings.simplefilter('ignore',RuntimeWarning);scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)
 return np.sort(np.argsort(-scores,kind='stable')[:k])

class MissingBernoulli10:
 def __init__(self,alpha=.1,pooled=False,fit_prior=False,k=None):self.alpha=alpha;self.pooled=pooled;self.fit_prior=fit_prior;self.k=k
 def fit(self,a,target,missing,sample_weight=None):
  self.cols=select_cols10(a,target,self.k);a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols]);w=np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)
  self.classes_=np.unique(target);nclass=np.array([w[target==c].sum() for c in self.classes_])
  counts=np.vstack([np.asarray(a[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  absent=np.vstack([np.asarray(missing[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  observed=nclass[:,None]-absent
  if self.pooled:
   denominator=w.sum()-np.asarray(missing.multiply(w[:,None]).sum(axis=0)).ravel()
   prior=np.clip(np.asarray(a.multiply(w[:,None]).sum(axis=0)).ravel()/np.maximum(denominator,1e-9),1e-7,1-1e-7)
   prob=(counts+self.alpha*prior)/(observed+self.alpha)
  else:prob=(counts+self.alpha)/(observed+2*self.alpha)
  prob=np.clip(prob,1e-7,1-1e-7);self.log0=np.log1p(-prob);self.logit=np.log(prob)-self.log0
  prior=nclass/nclass.sum() if self.fit_prior else np.full(len(nclass),1/len(nclass));self.logprior=np.log(prior)
  self.coef_=self.logit;return self
 def predict_proba(self,a,missing):
  a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols])
  score=np.asarray(a@self.logit.T)+self.log0.sum(axis=1)+self.logprior-np.asarray(missing@self.log0.T)
  return softmax(score,axis=1)

class GroupSigmoidSVM10:
 def __init__(self,C=.1,k=2048,balanced_calibration=False):self.C=C;self.k=k;self.balanced_calibration=balanced_calibration
 def fit(self,a,target,profile_groups,sample_weight):
  self.classes_=np.unique(target);oof=np.zeros((len(target),len(self.classes_)),float)
  for tr,va in StratifiedGroupKFold(3,shuffle=True,random_state=310).split(a,target,profile_groups):
   assert not set(profile_groups[tr]).intersection(profile_groups[va])
   cols=select_cols10(a[tr],target[tr],self.k);scaler=MaxAbsScaler().fit(a[tr][:,cols])
   est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(scaler.transform(a[tr][:,cols])),target[tr],sample_weight=sample_weight[tr])
   scores=est.decision_function(csr32(scaler.transform(a[va][:,cols])))
   assert np.array_equal(est.classes_,self.classes_);oof[va]=scores
  self.calibrators=[]
  for j,c in enumerate(self.classes_):
   cal=LogisticRegression(C=1.,class_weight='balanced' if self.balanced_calibration else None,max_iter=1000).fit(oof[:,j,None],(target==c).astype(int),sample_weight=sample_weight);self.calibrators.append(cal)
  self.cols=select_cols10(a,target,self.k);self.scaler=MaxAbsScaler().fit(a[:,self.cols]);self.est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(self.scaler.transform(a[:,self.cols])),target,sample_weight=sample_weight);self.coef_=self.est.coef_;return self
 def predict_proba(self,a):
  score=self.est.decision_function(csr32(self.scaler.transform(a[:,self.cols])))
  p=np.column_stack([cal.predict_proba(score[:,j,None])[:,1] for j,cal in enumerate(self.calibrators)])
  return p/np.maximum(p.sum(axis=1,keepdims=True),1e-12)

class Member10:
 def __init__(self,name,space,est,representation,numeric,cols=None,scaler=None):
  self.name=name;self.view='v10_'+representation;self.representation=representation;self.numeric=numeric;self.vec=space.compact_vec(np.arange(len(space.names)));self.est=est;self.cols=est.cols if cols is None else cols;self.raw_names=space.names[self.cols];self.names=self.raw_names;self.scaler=scaler
  lookup={g:i for i,g in enumerate(genes)};self.feature_gene_positions=np.array([lookup.get(str(s).split(':')[1],-1) if ':' in s and not str(s).startswith('numeric:') else -1 for s in space.names])
 def missing_matrix(self,packet):
  out=np.zeros((len(packet['events']),len(self.feature_gene_positions)),bool);valid=self.feature_gene_positions>=0
  out[:,valid]=packet['missing'][:,self.feature_gene_positions[valid]]
  return csr32(sparse.csr_matrix(out))
 def predict_packet(self,packet,records=None):
  rs=records10(packet,self.representation,self.numeric) if records is None else records;a=csr32(self.vec.transform(rs))
  if isinstance(self.est,MissingBernoulli10):return self.est.predict_proba(a,self.missing_matrix(packet))
  if isinstance(self.est,GroupSigmoidSVM10):return self.est.predict_proba(a)
  tx=a[:,self.cols]
  if self.scaler is not None:tx=csr32(self.scaler.transform(tx))
  return self.est.predict_proba(tx)
 def predict_frame(self,frame):return self.predict_packet(packet10(frame[genes]))

predict_members6=predict_members
def predict_members(members,weights,frame):
 old=[(m,w) for m,w in zip(members,weights) if not hasattr(m,'predict_frame')];p=predict_members6([m for m,w in old],[w for m,w in old],frame) if old else None
 packet=None
 for m,w in zip(members,weights):
  if hasattr(m,'predict_frame'):
   if packet is None:packet=packet10(frame[genes])
   q=m.predict_packet(packet)*w;p=q if p is None else p+q
 return p

C10={
 'gene_nb_a0.1':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_a1':dict(rep='gene',algo='nb',alpha=1.,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_empirical_prior':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=True,k=None,mask=0.),
 'gene_nb_pooled20':dict(rep='gene',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=None,mask=0.),
 'type_nb_pooled20':dict(rep='type',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=4096,mask=0.),
 'gene_lr':dict(rep='gene',algo='lr',C=.1,k=2048,mask=0.),
 'gene_lr_mask10':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.1),
 'gene_lr_mask25':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.25),
 'type_lr':dict(rep='type',algo='lr',C=.1,k=2048,mask=0.),
 'type_lr_mask10':dict(rep='type',algo='lr',C=.1,k=2048,mask=.1),
 'functional_lr':dict(rep='functional',algo='lr',C=.1,k=2048,mask=0.),
 'functional_lr_mask10':dict(rep='functional',algo='lr',C=.1,k=2048,mask=.1),
 'gene_svm':dict(rep='gene',algo='svm',C=.1,k=2048,mask=0.),
 'type_svm':dict(rep='type',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm':dict(rep='functional',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm_mask10':dict(rep='functional',algo='svm',C=.1,k=2048,mask=.1),
 'functional_lgb':dict(rep='functional',algo='lgb',k=2048,mask=0.),
 'functional_lgb_mask10':dict(rep='functional',algo='lgb',k=2048,mask=.1),
 'functional_lgb_mask25':dict(rep='functional',algo='lgb',k=2048,mask=.25)}

def prepare10(idx,representation,numeric,cache):
 key=(representation,numeric)
 if key not in cache:
  if '_clean' not in cache:cache['_clean']=packet10(X.iloc[idx],[events[i] for i in idx])
  rs=records10(cache['_clean'],representation,numeric);space=Space();a=space.fit(rs);cache[key]=(space,a)
 return cache[key]

def train10(name,idx,cache):
 cfg=C10[name];numeric=cfg['algo'] not in ['nb','cnb'];space,a=prepare10(idx,cfg['rep'],numeric,cache);target=yi[idx];rw=np.ones(len(idx));train_groups=groups[idx]
 augmented=None
 if cfg['mask']:
  key=('masked',cfg['mask'])
  if key not in cache:cache[key]=mask_packet10(cache['_clean'],cfg['mask'],1042+int(cfg['mask']*100))
  augmented=cache[key];ar=records10(augmented,cfg['rep'],numeric);a_aug=space.transform(ar);a=csr32(sparse.vstack([a,a_aug],format='csr'));target=np.r_[target,target];train_groups=np.r_[train_groups,train_groups];rw=np.r_[rw,.5*rw]/1.5
 if cfg['algo']=='nb':
  est=MissingBernoulli10(cfg['alpha'],cfg['pooled'],cfg['fit_prior'],cfg['k'])
  temp=Member10(name,space,est,cfg['rep'],False,cols=np.arange(a.shape[1]));missing=temp.missing_matrix(cache['_clean']);est.fit(a,target,missing,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False)
 if cfg['algo']=='cnb':
  cols=select_cols10(a,target,cfg['k']);est=ComplementNB(alpha=cfg['alpha'],norm=False).fit(a[:,cols],target,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False,cols=cols)
 if cfg['algo']=='svm':
  est=GroupSigmoidSVM10(cfg['C'],cfg['k'],cfg.get('balanced_calibration',False)).fit(a,target,train_groups,rw);return Member10(name,space,est,cfg['rep'],True)
 # Outer selection and scaler are fit on the original training patients only.
 cols=select_cols10(prepare10(idx,cfg['rep'],numeric,cache)[1],yi[idx],cfg['k'])
 scaler=MaxAbsScaler().fit(prepare10(idx,cfg['rep'],numeric,cache)[1][:,cols]) if cfg['algo']=='lr' else None
 tx=a[:,cols];tx=csr32(scaler.transform(tx)) if scaler is not None else csr32(tx)
 est=LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000) if cfg['algo']=='lr' else LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=42)
 est.fit(tx,target,sample_weight=rw);return Member10(name,space,est,cfg['rep'],True,cols=cols,scaler=scaler)

# A missing feature must contribute no Bernoulli presence or absence evidence.
toy=sparse.csr_matrix([[1,0],[1,0],[0,1],[0,1]],dtype=np.float32);toy_m=sparse.csr_matrix(toy.shape,dtype=np.float32)
nbtoy=MissingBernoulli10(alpha=1.).fit(toy,np.array([0,0,1,1]),toy_m)
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[1,0]],dtype=np.float32),sparse.csr_matrix([[0,0]],dtype=np.float32)),[[.9,.1]])
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[0,0]],dtype=np.float32),sparse.csr_matrix([[1,1]],dtype=np.float32)),[[.5,.5]])
assert all(kind10(t)=='synonymous' for t in expand10('1_2FS>FS'))
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2));print('V10 candidates:',len(C10),'semantic checks passed')

C10.update({'functional_svm_balanced': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.0, 'balanced_calibration': True}, 'functional_svm_balanced_mask10': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.1, 'balanced_calibration': True}, 'type_complement_nb_a0.1': {'rep': 'type', 'algo': 'cnb', 'alpha': 0.1, 'k': 4096, 'mask': 0.0}, 'type_complement_nb_a1': {'rep': 'type', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}, 'functional_complement_nb_a1': {'rep': 'functional', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}})
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2))
print('Extended candidates:',len(C10))


V10 candidates: 19 semantic checks passed
Extended candidates: 24


In [8]:
# Freeze the ablation and adoption rules before scoring new candidates.
PAIRS15=[('GBMLGG','LGG'),('KIPAN','KIRC')]
CONFIG15={f'{rep}_aug{aug}':dict(rep=rep,aug=aug) for rep in ['raw','gene','type'] for aug in [0,10]}
PROTO15={'pairs':PAIRS15,'configs':CONFIG15,'alpha':.5,'C':.1,'k':512,'class_weight':None,'augmentation_weight_ratio':[1.,.5],'vocab_selection_scaler_fit':'clean pair training only','noise_pool':'all outer training patients, never validation','primary_seed':43,'confirmation_seeds':[109,211,313],'reserved_fresh_seed_if_pass':509,'primary_clean_gain_vs_v10':.004,'confirmation_mean_clean_gain':.003,'each_confirmation_clean_floor':0.,'each_confirmation_noise_floor':-.003,'fresh_mean_gain':.003,'test_policy':'no test/sample access','holdout_policy':'no scoring','submission_policy':'do not create CSV','patient_gating':False}
(OUT/'protocol.json').write_text(json.dumps(PROTO15,indent=2))
sha15=hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest()
def rs15(ev,rep):
 records=EvidenceEncoder(1.,1.).fit([]).transform(ev)
 if rep=='raw':return records
 prefixes=('gene:','numeric:','module:') if rep=='gene' else ('gene:','type:','numeric:','module:')
 return [{k:v for k,v in d.items() if k.startswith(prefixes)} for d in records]
class Head15:
 def __init__(self,pair,rep,space,cols,scaler,est):self.pair=pair;self.rep=rep;self.vec=space.compact_vec(cols);self.names=space.names[cols];self.scaler=scaler;self.est=est
 def predict_records(self,records):return self.est.predict_proba(csr32(self.scaler.transform(self.vec.transform(records))))[:,1]
 def predict_frame(self,frame):return self.predict_records(rs15(parse_events(frame[genes]),self.rep))
def fit15(tr,pair,cfg,cache,seed):
 idx=tr[np.isin(y[tr],pair)];target=(y[idx]==pair[1]).astype(int);key=(tuple(pair),cfg['rep'])
 if key not in cache:
  records=rs15([events[i] for i in idx],cfg['rep']);space=Space();a=space.fit(records);cols=select_cols10(a,target,512);scaler=MaxAbsScaler().fit(a[:,cols]);cache[key]=(space,cols,scaler,csr32(a[:,cols]))
 space,cols,scaler,a=cache[key];weights=np.ones(len(idx));targets=target
 if cfg['aug']:
  nkey=('augframe',tuple(pair),cfg['aug'])
  if nkey not in cache:
   background=rare_background(X.iloc[idx],[events[i] for i in tr],seed,count=cfg['aug']);cache[nkey]=parse_events(background)
  ar=rs15(cache[nkey],cfg['rep']);a=sparse.vstack([a,space.transform(ar)[:,cols]],format='csr');targets=np.r_[target,target];weights=np.r_[weights,.5*weights]/1.5
 est=LogisticRegression(C=.1,max_iter=1500).fit(csr32(scaler.transform(a)),targets,sample_weight=weights)
 return Head15(pair,cfg['rep'],space,cols,scaler,est)
def apply15(p,conditional):
 out=p.copy()
 for j,pair in enumerate(PAIRS15):
  left,right=[int(np.flatnonzero(labels==c)[0]) for c in pair];mass=p[:,left]+p[:,right];old=np.divide(p[:,right],mass,out=np.full(len(p),.5),where=mass>0);new=.5*old+.5*conditional[j];out[:,right]=mass*new;out[:,left]=mass*(1-new)
 assert np.isfinite(out).all() and np.allclose(out.sum(1),1,atol=1e-5);return out
# Validated baseline caches are optional. A cache-free notebook refits the same v10 models.
group_path=WORK/'artifacts/v14/canonical_groups.npz';proof_path=WORK/'artifacts/v14/verification.json'
if group_path.exists() and proof_path.exists():
 proof=json.loads(proof_path.read_text());assert proof['train_sha256']==sha15
 q=np.load(group_path);assert np.array_equal(q['dev'],dev) and np.array_equal(q['hold'],hold);groups15=q['groups']
else:
 raw=X.fillna('__MISSING__').astype(str).copy()
 for g in genes:
  mask=~raw[g].isin(['WT','__MISSING__'])
  if mask.any():raw.loc[mask,g]=raw.loc[mask,g].map(lambda s:' '.join(sorted(set(canon(t) for t in s.split()))))
 groups15=pd.util.hash_pandas_object(raw,index=False).to_numpy()
assert not set(groups15[dev]).intersection(groups15[hold])
SPLIT15={s:list(StratifiedGroupKFold(3,shuffle=True,random_state=s).split(X.iloc[dev],y[dev],(groups15 if s in [313,509] else groups)[dev])) for s in [43,109,211,313,509]}
for seed,splits in SPLIT15.items():
 for a,b in splits:assert not set(groups15[dev[a]]).intersection(groups15[dev[b]])
def reference15(seed):
 path=OUT/f'reference_{seed}.npz'
 if path.exists():
  q=np.load(path,allow_pickle=True);assert str(q['train_sha256'])==sha15 and np.array_equal(q['dev'],dev) and np.array_equal(q['labels'],labels);return q['probabilities']
 if seed in [43,109,211]:
  p=WORK/'artifacts/v10/verification.json';c=WORK/'artifacts/v10'/('oof_probabilities.npz' if seed==43 else 'confirmation_probabilities.npz')
  if p.exists() and c.exists():
   assert json.loads(p.read_text())['data_sha256']['train.csv']==sha15;q=np.load(c,allow_pickle=True);assert np.array_equal(q['dev'],dev) and np.array_equal(q['labels'],labels)
   base='v6_reference' if seed==43 else f's{seed}__v6_reference';nb='type_complement_nb_a0.1' if seed==43 else f's{seed}__type_complement_nb_a0.1';return .75*q[base]+.25*q[nb]
 if seed==313:
  p=WORK/'artifacts/v14/verification.json';c=WORK/'artifacts/v14/diagnostic_probabilities.npz'
  if p.exists() and c.exists():
   assert json.loads(p.read_text())['train_sha256']==sha15;q=np.load(c,allow_pickle=True);assert np.array_equal(q['dev'],dev) and np.array_equal(q['labels'],labels);return q['s313_v10'][None,:,:]
 out=np.zeros((3,len(dev),NC),np.float32)
 for f,(a,b) in enumerate(SPLIT15[seed]):
  tr=dev[a];frames,_,_=eval_views(dev[b],tr,42+901+f if seed==43 else 42+seed*10+f);ms,ws=fit_v6(tr);nb=train10('type_complement_nb_a0.1',tr,{})
  for j,frame in enumerate(frames):out[j,b]=.75*predict_members(ms,ws,frame)+.25*nb.predict_frame(frame)
  print('V15 reference refit',seed,f,flush=True)
 np.savez_compressed(path,train_sha256=sha15,dev=dev,labels=labels,probabilities=out);return out
REF15={43:reference15(43)}
PRIMARY15=np.zeros((len(CONFIG15),2,3,len(dev)),np.float32);feature_audit15=[];names15=list(CONFIG15)
for f,(a,b) in enumerate(SPLIT15[43]):
 tr=dev[a];va=dev[b];frames,evs,_=eval_views(va,tr,42+901+f);cache={};records={rep:[rs15(ev,rep) for ev in evs] for rep in ['raw','gene','type']}
 for ci,(name,cfg) in enumerate(CONFIG15.items()):
  for j,pair in enumerate(PAIRS15):
   m=fit15(tr,pair,cfg,cache,seed=15000+100*f+j)
   for view in range(3):PRIMARY15[ci,j,view,b]=m.predict_records(records[cfg['rep']][view])
   feature_audit15.append(dict(fold=f,config=name,pair='/'.join(pair),features=len(m.names),clean_pair_patients=int(np.isin(y[tr],pair).sum()),augmentation_mean=cfg['aug']))
  print('V15 primary',f,name,'complete',flush=True)
 np.savez_compressed(OUT/'primary_conditionals.npz',dev=dev,labels=labels,names=np.array(names15),conditional=PRIMARY15)
rows15=[]
for ci,name in enumerate(names15):
 for f,(a,b) in enumerate(SPLIT15[43]):
  vals=[];old=[]
  for view in range(3):
   p=REF15[43][view,b];q=apply15(p,PRIMARY15[ci,:,view][:,b]);vals.append(f1_score(yi[dev[b]],q.argmax(1),average='macro',zero_division=0));old.append(f1_score(yi[dev[b]],p.argmax(1),average='macro',zero_division=0))
  rows15.append(dict(config=name,fold=f,clean=vals[0],noise=vals[1],duplicate=vals[2],score=.6*vals[0]+.3*vals[1]+.1*vals[2],gain_clean=vals[0]-old[0],gain_noise=vals[1]-old[1]))
pd.DataFrame(feature_audit15).to_csv(OUT/'feature_audit.csv',index=False);pd.DataFrame(rows15).to_csv(OUT/'primary_folds.csv',index=False);rank15=pd.DataFrame(rows15).groupby('config').mean(numeric_only=True).sort_values('clean',ascending=False);rank15.to_csv(OUT/'primary_ranking.csv')
eligible15=rank15[(rank15.gain_clean>=.004)&(rank15.gain_noise>=-.003)];primary_winner15=None if eligible15.empty else eligible15.index[0]
# Freeze at most two diagnostics: primary best and best changed representation/augmentation.
frozen15=list(dict.fromkeys([rank15.index[0],next(name for name in rank15.index if name!='raw_aug0')]))
(OUT/'selection_primary.json').write_text(json.dumps({'eligible':eligible15.index.tolist(),'winner':primary_winner15,'diagnostic_frozen':frozen15,'diagnostics_cannot_override_primary_gate':True},indent=2));print('V15 PRIMARY',rank15.to_string(),flush=True);print('FROZEN',frozen15,flush=True)
DIAG15={};diag_rows15=[]
for seed in [109,211,313]:
 REF15[seed]=reference15(seed);pviews=REF15[seed].shape[0];probs={name:np.zeros((pviews,len(dev),NC),np.float32) for name in frozen15}
 for f,(a,b) in enumerate(SPLIT15[seed]):
  tr=dev[a];va=dev[b];frames,evs,_=eval_views(va,tr,42+seed*10+f);cache={};records={rep:[rs15(ev,rep) for ev in evs[:pviews]] for rep in {CONFIG15[name]['rep'] for name in frozen15}}
  for name in frozen15:
   cond=np.zeros((2,pviews,len(b)),np.float32);cfg=CONFIG15[name]
   for j,pair in enumerate(PAIRS15):
    m=fit15(tr,pair,cfg,cache,seed=15000+100*seed+10*f+j)
    for v in range(pviews):cond[j,v]=m.predict_records(records[cfg['rep']][v])
   for v in range(pviews):
    p=REF15[seed][v,b];q=apply15(p,cond[:,v]);probs[name][v,b]=q;old=f1_score(yi[dev[b]],p.argmax(1),average='macro',zero_division=0);new=f1_score(yi[dev[b]],q.argmax(1),average='macro',zero_division=0);diag_rows15.append(dict(seed=seed,fold=f,config=name,view=['clean','noise','duplicate'][v],v10_f1=old,candidate_f1=new,delta_f1=new-old))
  pd.DataFrame(diag_rows15).to_csv(OUT/'confirmation_folds.csv',index=False);print('V15 frozen diagnostics',seed,f,'complete',flush=True)
 DIAG15[seed]=probs;np.savez_compressed(OUT/'confirmation_probabilities.npz',dev=dev,labels=labels,**{f's{s}__{n}':a for s,p in DIAG15.items() for n,a in p.items()})
summary15=pd.DataFrame(diag_rows15).groupby(['config','seed','view'])[['v10_f1','candidate_f1','delta_f1']].mean();summary15.to_csv(OUT/'confirmation_summary.csv')
accepted15=False;fresh_rows15=[]
if primary_winner15 is not None:
 checks=summary15.loc[primary_winner15];clean=checks.xs('clean',level='view').delta_f1;noise=checks.xs('noise',level='view').delta_f1
 passed=bool(clean.mean()>=.003 and (clean>=0).all() and (noise>=-.003).all())
 if passed:
  seed=509;ref=reference15(seed);cfg=CONFIG15[primary_winner15]
  for f,(a,b) in enumerate(SPLIT15[seed]):
   tr=dev[a];frames,evs,_=eval_views(dev[b],tr,42+seed*10+f);cache={};cond=np.zeros((2,3,len(b)))
   for j,pair in enumerate(PAIRS15):
    m=fit15(tr,pair,cfg,cache,seed=15000+100*seed+10*f+j)
    for v in range(3):cond[j,v]=m.predict_records(rs15(evs[v],cfg['rep']))
   for v in range(3):
    old=f1_score(yi[dev[b]],ref[v,b].argmax(1),average='macro');new=f1_score(yi[dev[b]],apply15(ref[v,b],cond[:,v]).argmax(1),average='macro');fresh_rows15.append(dict(fold=f,view=['clean','noise','duplicate'][v],old=old,new=new,delta=new-old))
  fresh=pd.DataFrame(fresh_rows15);fresh.to_csv(OUT/'reserved_seed509.csv',index=False);accepted15=bool(fresh[fresh.view=='clean'].delta.mean()>=.003 and fresh[fresh.view=='noise'].delta.mean()>=-.003)
selection15={'accepted':accepted15,'primary_winner':primary_winner15,'frozen_diagnostics':frozen15,'primary_eligible':eligible15.index.tolist(),'fresh509_run':bool(fresh_rows15),'holdout_scored':False,'submission_created':False}
(OUT/'selection.json').write_text(json.dumps(selection15,indent=2));print('V15 CONFIRMATION',summary15.to_string(),flush=True);print('V15 SELECTION',selection15,flush=True)
if accepted15:
 # Save a train-only model only after every predeclared validation gate passes. Never infer test here.
 base_path=WORK/'artifacts/v10/cancer_model_v10_clean_experimental.pkl';proof_path=WORK/'artifacts/v10/verification.json'
 if base_path.exists() and proof_path.exists():
  assert json.loads(proof_path.read_text())['data_sha256']['train.csv']==sha15
  with base_path.open('rb') as f:base=cloudpickle.load(f)
 else:
  ms,ws=fit_v6(np.arange(len(train)));nb=train10('type_complement_nb_a0.1',np.arange(len(train)),{});base={'models':ms+[nb],'weights':np.r_[ws*.75,.25],'genes':genes,'classes':labels}
  def base_predict15(b,frame):return predict_members(b['models'],b['weights'],frame)
  base['predict']=base_predict15
 heads=[fit15(np.arange(len(train)),p,CONFIG15[primary_winner15],{},seed=15151+j) for j,p in enumerate(PAIRS15)]
 def predict15(b,frame):return apply15(b['base']['predict'](b['base'],frame),np.stack([m.predict_frame(frame) for m in b['heads']]))
 model15={'base':base,'heads':heads,'predict':predict15,'genes':genes,'labels':labels,'selection':selection15}
 with (OUT/'cancer_model_v15.pkl').open('wb') as f:cloudpickle.dump(model15,f)
print('V15 training and diagnostics complete; inference inputs remain unopened.',flush=True)


V15 primary 0 raw_aug0 complete
V15 primary 0 raw_aug10 complete
V15 primary 0 gene_aug0 complete
V15 primary 0 gene_aug10 complete
V15 primary 0 type_aug0 complete
V15 primary 0 type_aug10 complete
V15 primary 1 raw_aug0 complete
V15 primary 1 raw_aug10 complete
V15 primary 1 gene_aug0 complete
V15 primary 1 gene_aug10 complete
V15 primary 1 type_aug0 complete
V15 primary 1 type_aug10 complete
V15 primary 2 raw_aug0 complete
V15 primary 2 raw_aug10 complete
V15 primary 2 gene_aug0 complete
V15 primary 2 gene_aug10 complete
V15 primary 2 type_aug0 complete
V15 primary 2 type_aug10 complete
V15 PRIMARY             fold     clean     noise  duplicate     score  gain_clean  gain_noise
config                                                                           
gene_aug10   1.0  0.445842  0.344177   0.443542  0.415112    0.003437   -0.001816
type_aug10   1.0  0.445176  0.343556   0.443091  0.414482    0.002772   -0.002437
gene_aug0    1.0  0.445082  0.342496   0.443006  0.414099    0.

## 분석 보고서·입력 설명·검증 증거 저장

In [9]:
# Save evidence without reopening inference data or scoring the old holdout.
assert forbidden_attempts15==[]
rank15=pd.read_csv(OUT/'primary_ranking.csv',index_col='config');summary15=pd.read_csv(OUT/'confirmation_summary.csv').set_index(['config','seed','view']);selection15=json.loads((OUT/'selection.json').read_text());protocol15=json.loads((OUT/'protocol.json').read_text());audit15=pd.read_csv(OUT/'feature_audit.csv');v14proof=json.loads((WORK/'artifacts/v14/verification.json').read_text()) if (WORK/'artifacts/v14/verification.json').exists() else None
# The unchanged raw/no-augmentation control must reproduce the previous experiment.
if v14proof:
 assert v14proof['train_sha256']==hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest()
 assert abs(rank15.loc['raw_aug0','clean']-v14proof['best_pair_primary_f1'])<1e-10
previous15=pd.read_csv(WORK/'artifacts/v14/diagnostic_repeated_cv.csv') if (WORK/'artifacts/v14/diagnostic_repeated_cv.csv').exists() else None
if previous15 is not None:
 for (name,seed,view),r in summary15.iterrows():
  old=previous15[(previous15.seed==seed)&(previous15.view==view)]
  if len(old):assert abs(r.v10_f1-old.v10_f1.mean())<1e-10
ablation15=[]
for rep in ['raw','gene','type']:
 a=rank15.loc[f'{rep}_aug0'];b=rank15.loc[f'{rep}_aug10'];ablation15.append({'representation':rep,'clean_no_aug':a.clean,'clean_aug10':b.clean,'augmentation_delta_clean':b.clean-a.clean,'noise_no_aug':a.noise,'noise_aug10':b.noise,'augmentation_delta_noise':b.noise-a.noise})
pd.DataFrame(ablation15).to_csv(OUT/'ablation_effects.csv',index=False)
report15='# v15 — 쌍별 분류기 입력 단순화·학습 전용 희귀 변이 증강\n\n**결론: 사전 채택 기준 미달, 새 제출 권고 없음.** test/sample은 열지 않고 기존 홀드아웃도 재평가하지 않았습니다. 새 제출 CSV는 만들지 않았습니다. v10은 확인된 Public 기준 모델로 보관합니다.\n\n'
report15+='## 가설과 고정한 것\n\nv14는 원본 F1이 조금 올랐으나 seed 211의 합성 노이즈 F1이 0.006179 낮아졌습니다. 정확한 변이/위치에 대한 의존을 줄이고 학습 부분에만 희귀 변이를 추가하면 이 취약성을 줄일 수 있는지 비교했습니다. GBMLGG/LGG, KIPAN/KIRC 두 쌍, empirical LR C=0.1, chi2 최대 512개, 고정 alpha=0.5는 유지했습니다. 환자별 혼합 비중/게이팅은 사용하지 않았습니다.\n\n'
report15+='## 여섯 개 대조 실험\n\nraw/gene/type × 증강 없음/Poisson 평균 10개 주입입니다. gene은 gene·numeric·module, type은 gene·type·numeric·module를 남깁니다. 숫자와 경로 요약은 모든 표현에서 유지하므로 gene 입력을 순수 유전자 이진값만으로 설명하지 않습니다. 같은 파서·정규화·LR·선별 예산을 사용했습니다.\n\n희귀 변이 풀은 해당 외부 fold 학습 환자 전체에서 관측 환자 2명 이하인 동의/치환 변이를 모아 두 종류를 50:50으로 샘플링했습니다. 검증 환자는 풀 생성에서 제외했고 두 암종 모두 같은 풀을 사용했습니다. 증강본은 원본 학습 환자에게만 붙였으며 별도 분할로 이동하지 않았습니다. 어휘·빈도·chi2·MaxAbs는 원본 쌍 학습 부분에서만 적합했습니다. 증강에서만 나온 열을 추가하지 않았습니다. 원본/증강 가중치 1:0.5를 합계가 원본 환자 수가 되도록 정규화해 정규화 강도가 표본 수 증가만으로 바뀌지 않게 했습니다.\n\n이것은 라벨 보존이 생물학적으로 확정된 변환이 아니라 합성 가정입니다. 평가 noise 평균 20개와 학습 noise 평균 10개의 난수 seed는 분리했습니다.\n\n'
report15+='## 1차 그룹 CV\n\n'+rank15.drop(columns='fold',errors='ignore').to_markdown()+'\n\n각 gain은 같은 fold의 v10 대비 변화입니다. v10 0.442404, v14/raw_aug0 0.444642를 비교 기준으로 사용했습니다. raw_aug0 재현과 각 확인 분할의 baseline 일치를 검증했습니다.\n\n'
report15+='## 증강 효과 분리\n\n'+pd.DataFrame(ablation15).to_markdown(index=False)+'\n\n원본 입력에서의 증강과 단순화된 입력에서의 증강이 같은 효과를 내지는 않았습니다. 양의 변화가 있다는 이유만으로 전체 노이즈 문제가 해결됐다고 보지 않습니다.\n\n'
report15+='## 고정 후보의 확인 분할\n\n'+summary15.to_markdown()+'\n\n109/211/313은 같은 개발 환자의 재분할이므로 독립 외부 환자 검증이 아닙니다. seed 313에는 검증된 기존 원본 baseline만 있어 원본만 비교했고 stress 점수를 만들어 넣지 않았습니다. 최고 후보와 변경 설정 중 최고를 최대 두 개까지 1차 결과로 고정했으며 확인 결과를 보고 파라미터를 다시 바꾸지 않았습니다.\n\n'
report15+='## 채택 기준과 결과\n\n1차 원본 +0.004, 확인 분할 평균 +0.003, 각 seed 원본 비하락 및 stress 하락 0.003 이하를 실험 전에 고정했습니다. 이를 모두 통과한 경우에만 새로운 seed 509의 마지막 감사를 실행하도록 했습니다. 기준을 통과하지 못한 진단으로 1차 탈락을 뒤집지 않습니다.\n\n'+json.dumps(selection15,ensure_ascii=False,indent=2)+'\n\n'
if not selection15['accepted']:
 report15+='새 후보를 채택하지 않았고, 509 감사·전체 train 최종 모델 학습·제출 파일 생성은 수행하지 않았습니다. 기존 주 모델과 v10 제출 파일은 유지했습니다. 기존 홀드아웃의 새 점수나 Public 점수는 없습니다.\n\n'
report15+='## 분리·재현\n\n고정 원본 그룹 및 정규화 프로필이 각 학습/검증 양쪽에 걸치지 않음을 확인했습니다. baseline cache는 train 해시·개발 행·클래스 및 기존 재분할 점수를 확인한 뒤 재사용했습니다. 캐시가 없으면 같은 학습 부분으로 baseline도 이 노트북 안에서 재학습합니다. 외부 검증을 조기 종료나 feature fitting에 사용하지 않았습니다. 과거 버전에서 test EDA와 반복 홀드아웃을 관찰한 설계 편향이 없어졌다는 주장은 하지 않습니다.\n'
(WORK/'analysis_report_v15.md').write_text(report15)
(WORK/'selected_variables_v15.md').write_text('# v15 입력 변수와 선별 근거\n\nraw는 기존 유전자·변이·위치·아미노산 변화·유형·숫자·경로 표현입니다. gene은 gene:/numeric:/module: 접두사만, type은 여기에 type:을 추가한 표현입니다. 숫자 요약에는 변이 수·유형별 비율도 들어가므로 순수 이진 gene-only 모델이 아닙니다. 정확한 표기에 대한 의존을 줄이려는 가설이며 정보 손실로 성능이 떨어질 수도 있습니다.\n\n같은 외부 학습 fold의 해당 두 암종 환자 원본만으로 관측 빈도 3 이상 어휘를 만들고 binary chi2 최대 512개를 선별합니다. MaxAbs도 같은 원본 행렬에 적합합니다. 증강본이나 검증 환자로 어휘/선별/스케일을 재적합하지 않습니다. 각 head와 fold의 실제 입력 수는 아래와 같습니다.\n\n'+audit15.to_markdown(index=False)+'\n\n최종 제출 후보가 없으므로 전체 train 최종 선별 목록은 새로 만들지 않았습니다. 각 fold의 실험 변수는 그 fold 학습 부분에서 결정되며 서로 같다는 보장은 없습니다. 기존 v10 최종 목록은 selected_variables_v10.md를 참조합니다.\n')
proof15={'status':'passed','version':15,'accepted_new_candidate':selection15['accepted'],'selection':selection15,'primary_best_config':rank15.index[0],'primary_best_macro_f1':float(rank15.iloc[0].clean),'primary_best_gain_vs_v10':float(rank15.iloc[0].gain_clean),'test_opened':False,'sample_opened':False,'holdout_scored':False,'submission_created':False,'forbidden_attempts':forbidden_attempts15,'train_sha256':hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest(),'control_v14_reproduced':v14proof is not None,'reference_fold_scores_checked':previous15 is not None,'confirmation_summary':summary15.reset_index().to_dict('records'),'primary_heads_fitted':len(audit15),'daily_submission_used':False}
(OUT/'verification.json').write_text(json.dumps(proof15,indent=2));print(json.dumps(proof15,indent=2),flush=True)


{
  "status": "passed",
  "version": 15,
  "accepted_new_candidate": false,
  "selection": {
    "accepted": false,
    "primary_winner": null,
    "frozen_diagnostics": [
      "gene_aug10"
    ],
    "primary_eligible": [],
    "fresh509_run": false,
    "holdout_scored": false,
    "submission_created": false
  },
  "primary_best_config": "gene_aug10",
  "primary_best_macro_f1": 0.44584183939792355,
  "primary_best_gain_vs_v10": 0.003437370205248517,
  "test_opened": false,
  "sample_opened": false,
  "holdout_scored": false,
  "submission_created": false,
  "forbidden_attempts": [],
  "train_sha256": "92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab",
  "control_v14_reproduced": true,
  "reference_fold_scores_checked": true,
  "confirmation_summary": [
    {
      "config": "gene_aug10",
      "seed": 109,
      "view": "clean",
      "v10_f1": 0.43653660121133986,
      "candidate_f1": 0.4385424791737241,
      "delta_f1": 0.0020058779623842358
    },
    {
      "